## Imports & Config

In [1]:
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

2026-03-01 16:36:27.956677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772382988.148036      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772382988.206219      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772382988.652091      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772382988.652139      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772382988.652142      55 computation_placer.cc:177] computation placer alr

In [2]:
data_dir="/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"

IMG_SIZE=224
BATCH_SIZE=32
SEED=42

## Data Pipeline

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

temp_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

temp_batches=tf.data.experimental.cardinality(temp_ds)
val_size=temp_batches//2

val_ds=temp_ds.take(val_size)
test_ds=temp_ds.skip(val_size)

Found 54305 files belonging to 38 classes.
Using 38014 files for training.


I0000 00:00:1772383044.384585      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 54305 files belonging to 38 classes.
Using 16291 files for validation.


## Preprocessing

In [4]:
from tensorflow.keras.applications.efficientnet import preprocess_input

train_ds=train_ds.map(lambda x,y: (preprocess_input(x),y))
val_ds=val_ds.map(lambda x,y: (preprocess_input(x),y))
test_ds=test_ds.map(lambda x,y: (preprocess_input(x),y))

## Data Augmentation

In [5]:
data_augmentation=tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds=train_ds.map(
    lambda x,y:(data_augmentation(x,training=True),y)
)

## Prefetch

In [6]:
AUTOTUNE=tf.data.AUTOTUNE

train_ds=train_ds.prefetch(AUTOTUNE)
val_ds=val_ds.prefetch(AUTOTUNE)
test_ds=test_ds.prefetch(AUTOTUNE)

## Load EfficientNet Backbone

In [7]:
base_model=EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Freeze Backbone

In [8]:
base_model.trainable=False

## Build Full Model

In [9]:
num_classes=38

model=models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(num_classes,activation='softmax')
])

## Compile

In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## Callbacks

In [11]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

checkpoint = ModelCheckpoint(
    "/kaggle/working/efficientnet_frozen.keras",
    save_best_only=True
)

In [13]:
history=model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop,reduce_lr,checkpoint]
)

Epoch 1/15


I0000 00:00:1772383100.906346     198 service.cc:152] XLA service 0x7d613c001940 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1772383100.906383     198 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1772383103.024567     198 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772383113.452015     198 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1188/1188 ━━━━━━━━━━━━━━━━━━━━ 432s 345ms/step - accuracy: 0.6930 - loss: 1.1990 - val_accuracy: 0.9430 - val_loss: 0.1823 - learning_rate: 0.0010
Epoch 2/15
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 384s 323ms/step - accuracy: 0.9030 - loss: 0.3088 - val_accuracy: 0.9484 - val_loss: 0.1649 - learning_rate: 0.0010
Epoch 3/15
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 382s 322ms/step - accuracy: 0.9161 - loss: 0.2688 - val_accuracy: 0.9571 - val_loss: 0.1386 - learning_rate: 0.0010
Epoch 4/15
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 373s 314ms/step - accuracy: 0.9232 - loss: 0.2552 - val_accuracy: 0.9585 - val_loss: 0.1353 - learning_rate: 0.0010
Epoch 5/15
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 379s 319ms/step - accuracy: 0.9252 - loss: 0.2452 - val_accuracy: 0.9623 - val_loss: 0.1270 - learning_rate: 0.0010
Epoch 6/15
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 383s 322ms/step - accuracy: 0.9275 - loss: 0.2476 - val_accuracy: 0.9631 - val_loss: 0.1201 - learning_rate: 0.0010
Epoch 7/15
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 380s 320ms/step - acc

## Unfreeze Top Layers

In [14]:
base_model.trainable=True

fine_tune_at=int(len(base_model.layers)*0.7)

for layers in base_model.layers[:fine_tune_at]:
    layers.trainable=False

## Recompile with Lower Learning Rate

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## Train Fine-Tuning Phase

In [16]:
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/10


2026-03-01 18:18:46.484322: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-01 18:18:46.691165: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-01 18:18:47.053183: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-01 18:18:47.259912: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1187/1188 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.7497 - loss: 0.9753

2026-03-01 18:25:14.720336: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-01 18:25:14.928177: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-01 18:25:15.287481: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-01 18:25:15.495246: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1188/1188 ━━━━━━━━━━━━━━━━━━━━ 434s 338ms/step - accuracy: 0.7498 - loss: 0.9748 - val_accuracy: 0.9382 - val_loss: 0.1867 - learning_rate: 1.0000e-05
Epoch 2/10
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 379s 319ms/step - accuracy: 0.8805 - loss: 0.4075 - val_accuracy: 0.9572 - val_loss: 0.1368 - learning_rate: 1.0000e-05
Epoch 3/10
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 380s 320ms/step - accuracy: 0.9122 - loss: 0.2996 - val_accuracy: 0.9653 - val_loss: 0.1059 - learning_rate: 1.0000e-05
Epoch 4/10
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 377s 317ms/step - accuracy: 0.9312 - loss: 0.2175 - val_accuracy: 0.9713 - val_loss: 0.0925 - learning_rate: 1.0000e-05
Epoch 5/10
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 389s 327ms/step - accuracy: 0.9420 - loss: 0.1805 - val_accuracy: 0.9732 - val_loss: 0.0874 - learning_rate: 1.0000e-05
Epoch 6/10
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 377s 317ms/step - accuracy: 0.9476 - loss: 0.1606 - val_accuracy: 0.9772 - val_loss: 0.0771 - learning_rate: 1.0000e-05
Epoch 7/10
1188/1188 ━━━━━━━━━━━━━━━━━━

In [17]:
test_loss, test_accuracy = model.evaluate(test_ds)
print("Fine-Tuned Test Accuracy:", test_accuracy)

255/255 ━━━━━━━━━━━━━━━━━━━━ 24s 77ms/step - accuracy: 0.9809 - loss: 0.0624
Fine-Tuned Test Accuracy: 0.9804452061653137


In [18]:
model.save("/kaggle/working/plant_disease_efficientnet_finetuned.keras")